# 07 — 3-Model Stack: Ridge Meta on XGB + LGBM + Extra Trees OOF

**Current best:** v7 stacking (XGB + LGBM) → Kaggle RMSE **21,905** | OOF RMSE **21,841**. Goal: push below 21,466 (1st place).

## Why add Extra Trees?

| Model | Type | How it builds trees |
|---|---|---|
| XGBoost | Boosting (sequential) | Optimises split thresholds to minimise loss |
| LightGBM | Boosting (sequential) | Same as XGB but leaf-wise growth |
| **Extra Trees** | **Bagging (parallel)** | **Random split thresholds** — no optimisation at all |

Because Extra Trees is fundamentally different (bagging vs boosting, random vs optimised splits), it makes **different errors** from XGB and LGBM.
When Ridge sees 3 OOF predictions instead of 2, it has more signal to correct mistakes.

Extra Trees is also explicitly listed in the **syllabus recap slide** — it's within competition scope.

## What changes vs notebook 06

| # | Change | Detail |
|---|---|---|
| 1 | **3rd base model** | Add `ExtraTreesRegressor` OOF alongside XGB + LGBM |
| 2 | **Wider meta-features** | Ridge now trains on `[xgb_oof, lgbm_oof, et_oof]` — 3 columns |
| 3 | **ET tuned** | RandomizedSearchCV for ET hyperparameters |

---
## 1. Imports & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from math import radians, cos, sin, asin, sqrt

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, make_scorer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

train = pd.read_csv('../../data/raw/train.csv', low_memory=False)
test  = pd.read_csv('../../data/raw/test.csv',  low_memory=False)
print(f'Train: {train.shape}  |  Test: {test.shape}')

---
## 2. Feature Engineering

Same as notebook 06 — all 11 engineered features including `street_freq`.

In [ ]:
DROP_COLS   = ['floor_area_sqft','lower','upper','mid','full_flat_type',
               'address','Tranc_YearMonth','residential','year_completed']
SOLD_COLS   = ['1room_sold','2room_sold','3room_sold','4room_sold',
               '5room_sold','exec_sold','multigen_sold','studio_apartment_sold']
RENTAL_COLS = ['1room_rental','2room_rental','3room_rental','other_room_rental']
FILL_ZERO   = ['Hawker_Within_500m','Mall_Within_500m','Hawker_Within_1km',
               'Mall_Within_1km','Hawker_Within_2km','Mall_Within_2km']
MATURE_ESTATES = {
    'ANG MO KIO','BEDOK','BISHAN','BUKIT MERAH','BUKIT TIMAH',
    'CENTRAL AREA','CLEMENTI','GEYLANG','KALLANG/WHAMPOA',
    'MARINE PARADE','PASIR RIS','QUEENSTOWN','SERANGOON','TAMPINES','TOA PAYOH'
}
ROOM_COUNT = {'1 ROOM':1,'2 ROOM':2,'3 ROOM':3,'4 ROOM':4,
              '5 ROOM':5,'EXECUTIVE':5,'MULTI-GENERATION':6}
CBD_LAT, CBD_LON = 1.2847, 103.8510

STREET_FREQ = train['street_name'].value_counts().to_dict()

def haversine_km(lat, lon, lat2=CBD_LAT, lon2=CBD_LON):
    R = 6371
    lat, lon, lat2, lon2 = map(radians, [lat, lon, lat2, lon2])
    a = sin((lat2-lat)/2)**2 + cos(lat)*cos(lat2)*sin((lon2-lon)/2)**2
    return 2*R*asin(sqrt(a))

def engineer_features(df, amenity_caps=None):
    df = df.copy()
    for c in FILL_ZERO:
        df[c] = df[c].fillna(0)
    df['remaining_lease']  = 99 - (df['Tranc_Year'] - df['lease_commence_date'])
    df['dist_to_cbd']      = df.apply(lambda r: haversine_km(r['Latitude'], r['Longitude']), axis=1)
    df['is_mature_estate'] = df['town'].str.upper().isin(MATURE_ESTATES).astype(int)
    df['tranc_month_sin']     = np.sin(2*np.pi*df['Tranc_Month']/12)
    df['tranc_month_cos']     = np.cos(2*np.pi*df['Tranc_Month']/12)
    df['total_sold']          = df[SOLD_COLS].sum(axis=1)
    df['total_rental']        = df[RENTAL_COLS].sum(axis=1)
    df['rental_ratio']        = (df['total_rental'] / df['total_dwelling_units'].replace(0, np.nan)).fillna(0)
    df['num_rooms']           = df['flat_type'].str.upper().map(ROOM_COUNT).fillna(4)
    df['floor_area_per_room'] = df['floor_area_sqm'] / df['num_rooms']
    caps = amenity_caps or {}
    new_caps = {}
    for col in ['mrt_nearest_distance','Mall_Nearest_Distance','Hawker_Nearest_Distance']:
        cap = caps.get(col, df[col].quantile(0.99))
        new_caps[col] = cap
        inv = 1 / df[col].clip(1, cap)
        inv_min, inv_max = inv.min(), inv.max()
        df[f'_inv_{col}'] = (inv - inv_min) / (inv_max - inv_min + 1e-9)
    df['amenity_score'] = (df['_inv_mrt_nearest_distance'] +
                           df['_inv_Mall_Nearest_Distance'] +
                           df['_inv_Hawker_Nearest_Distance']) / 3
    df.drop(columns=[c for c in df.columns if c.startswith('_inv_')], inplace=True)
    df['dist_x_storey']   = df['dist_to_cbd'] * df['mid_storey']
    df['lease_x_area']    = df['remaining_lease'] * df['floor_area_sqm']
    df['log_dist_to_cbd'] = np.log1p(df['dist_to_cbd'])
    df['street_freq']     = df['street_name'].map(STREET_FREQ).fillna(0)
    return df, new_caps

train_fe, train_caps = engineer_features(train)
test_fe,  _          = engineer_features(test, amenity_caps=train_caps)
print(f'Train: {train_fe.shape}  |  Test: {test_fe.shape}')

---
## 3. Per-Fold Target Encoding for `town_median_price`

In [ ]:
def oof_town_median(towns_series, y_series, n_splits=5, random_state=42):
    towns = towns_series.values
    y     = y_series.values
    encoded = np.zeros(len(towns))
    global_med = np.median(y)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for tr_idx, va_idx in kf.split(towns):
        fold_map = {}
        for t, p in zip(towns[tr_idx], y[tr_idx]):
            fold_map.setdefault(t, []).append(p)
        fold_map = {t: np.median(ps) for t, ps in fold_map.items()}
        for i in va_idx:
            encoded[i] = fold_map.get(towns[i], global_med)
    return encoded

train_fe['town_median_price'] = oof_town_median(
    train_fe['town'], train['resale_price']
)
full_town_map = train.groupby('town')['resale_price'].median()
test_fe['town_median_price'] = test_fe['town'].map(full_town_map).fillna(full_town_map.median())

print('OOF town encoding done.')

---
## 4. Prepare X and y

In [ ]:
DROP_ALL = ['id','resale_price'] + DROP_COLS + SOLD_COLS + RENTAL_COLS + ['num_rooms']

X      = train_fe.drop(columns=DROP_ALL, errors='ignore')
y      = train['resale_price'].values
y_log  = np.log1p(y)

X_test = test_fe.drop(columns=[c for c in DROP_ALL if c != 'resale_price'], errors='ignore')
X_test = X_test.reindex(columns=X.columns, fill_value=0)

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

for col in cat_cols:
    X[col]      = X[col].astype(str)
    X_test[col] = X_test[col].astype(str)

print(f'Features: {X.shape[1]}  (num={len(num_cols)}, cat={len(cat_cols)})')

---
## 5. Preprocessing Pipeline

In [ ]:
num_pipe = Pipeline([('imp', SimpleImputer(strategy='median'))])
cat_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

print('Preprocessor ready.')

---
## 6. Tune Base Models (RandomizedSearchCV)

Tune all **three** base models on an 80/20 split.

**Extra Trees hyperparameters to search:**
- `n_estimators` — more trees = more stable (diminishing returns above 500)
- `max_depth` — `None` = fully grown; shallow = less overfitting
- `min_samples_split` / `min_samples_leaf` — controls minimum node size
- `max_features` — fraction of features considered per split (0.5–0.8 typical for regression)

Extra Trees is faster to tune than XGB/LGBM because it uses random split thresholds.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
y_train_log = np.log1p(y_train)
y_val_log   = np.log1p(y_val)

def neg_dollar_rmse(y_log_true, y_log_pred):
    return -np.sqrt(mean_squared_error(np.expm1(y_log_true), np.expm1(y_log_pred)))

dollar_rmse_scorer = make_scorer(neg_dollar_rmse)

# --- XGBoost search ---
param_dist_xgb = {
    'model__n_estimators'     : [800, 1000, 1200, 1500],
    'model__max_depth'        : [5, 6, 7, 8],
    'model__learning_rate'    : [0.01, 0.03, 0.05, 0.08],
    'model__subsample'        : [0.6, 0.7, 0.8, 0.9],
    'model__colsample_bytree' : [0.5, 0.6, 0.7, 0.8],
    'model__min_child_weight' : [1, 3, 5],
    'model__reg_alpha'        : [0, 0.01, 0.1],
    'model__reg_lambda'       : [1, 1.5, 2],
}
xgb_search = RandomizedSearchCV(
    Pipeline([('pre', preprocessor), ('model', XGBRegressor(random_state=42, n_jobs=-1, verbosity=0, tree_method='hist'))]),
    param_distributions=param_dist_xgb, n_iter=30, cv=3,
    scoring=dollar_rmse_scorer, random_state=42, n_jobs=-1, verbose=1, refit=True,
)
xgb_search.fit(X_train, y_train_log)
print(f'\nXGB best CV RMSE: ${-xgb_search.best_score_:,.0f}')
xgb_val_pred = np.expm1(xgb_search.best_estimator_.predict(X_val))
print(f'XGB val RMSE:     ${rmse(y_val, xgb_val_pred):,.0f}')

# --- LightGBM search ---
param_dist_lgbm = {
    'model__n_estimators'     : [800, 1000, 1200, 1500],
    'model__max_depth'        : [6, 7, 8, 10],
    'model__num_leaves'       : [60, 80, 100, 127],
    'model__learning_rate'    : [0.01, 0.03, 0.05, 0.08],
    'model__subsample'        : [0.7, 0.8, 0.9],
    'model__colsample_bytree' : [0.6, 0.7, 0.8],
    'model__min_child_samples': [10, 20, 30],
    'model__reg_alpha'        : [0, 0.01, 0.1],
    'model__reg_lambda'       : [0, 0.1, 1],
}
lgbm_search = RandomizedSearchCV(
    Pipeline([('pre', preprocessor), ('model', LGBMRegressor(random_state=42, n_jobs=-1, verbosity=-1))]),
    param_distributions=param_dist_lgbm, n_iter=30, cv=3,
    scoring=dollar_rmse_scorer, random_state=42, n_jobs=-1, verbose=1, refit=True,
)
lgbm_search.fit(X_train, y_train_log)
print(f'\nLGBM best CV RMSE: ${-lgbm_search.best_score_:,.0f}')
lgbm_val_pred = np.expm1(lgbm_search.best_estimator_.predict(X_val))
print(f'LGBM val RMSE:     ${rmse(y_val, lgbm_val_pred):,.0f}')

# --- Extra Trees search ---
param_dist_et = {
    'model__n_estimators'    : [300, 500, 700, 1000],
    'model__max_depth'       : [None, 20, 30, 40],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features'    : [0.5, 0.6, 0.7, 0.8],
}
et_search = RandomizedSearchCV(
    Pipeline([('pre', preprocessor), ('model', ExtraTreesRegressor(random_state=42, n_jobs=-1))]),
    param_distributions=param_dist_et, n_iter=20, cv=3,
    scoring=dollar_rmse_scorer, random_state=42, n_jobs=-1, verbose=1, refit=True,
)
et_search.fit(X_train, y_train_log)
print(f'\nET best CV RMSE: ${-et_search.best_score_:,.0f}')
et_val_pred = np.expm1(et_search.best_estimator_.predict(X_val))
print(f'ET val RMSE:     ${rmse(y_val, et_val_pred):,.0f}')

# Extract best params
XGB_PARAMS  = {k.replace('model__',''):v for k,v in xgb_search.best_params_.items()}
LGBM_PARAMS = {k.replace('model__',''):v for k,v in lgbm_search.best_params_.items()}
ET_PARAMS   = {k.replace('model__',''):v for k,v in et_search.best_params_.items()}
XGB_PARAMS.update( {'random_state':42,'n_jobs':-1,'verbosity':0,'tree_method':'hist'})
LGBM_PARAMS.update({'random_state':42,'n_jobs':-1,'verbosity':-1})
ET_PARAMS.update(  {'random_state':42,'n_jobs':-1})

print(f'\nXGB_PARAMS:  {XGB_PARAMS}')
print(f'LGBM_PARAMS: {LGBM_PARAMS}')
print(f'ET_PARAMS:   {ET_PARAMS}')

---
## 7. Generate OOF Predictions (5-Fold) — 3 Models

Same pattern as notebook 06 but now running **three** base models per fold:
1. XGBoost
2. LightGBM
3. Extra Trees

Each fold produces OOF predictions for training rows and test predictions for later averaging.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

xgb_oof        = np.zeros(len(X))
lgbm_oof       = np.zeros(len(X))
et_oof         = np.zeros(len(X))

xgb_test_folds  = np.zeros((5, len(X_test)))
lgbm_test_folds = np.zeros((5, len(X_test)))
et_test_folds   = np.zeros((5, len(X_test)))

fold_rmses_xgb  = []
fold_rmses_lgbm = []
fold_rmses_et   = []

print(f'Generating OOF predictions (5-fold, 3 models)...')
print(f'{"Fold":>5}  {"XGB RMSE":>12}  {"LGBM RMSE":>12}  {"ET RMSE":>12}')
print('-' * 50)

for fold, (tr_idx, va_idx) in enumerate(kf.split(X)):
    X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
    y_tr, y_va = y[tr_idx], y[va_idx]
    y_tr_log   = np.log1p(y_tr)

    # Per-fold target encoding
    fold_town_map = {}
    for t, p in zip(X_tr['town'].values, y_tr):
        fold_town_map.setdefault(t, []).append(p)
    fold_town_map = {t: np.median(ps) for t, ps in fold_town_map.items()}
    global_med_tr = np.median(y_tr)
    X_tr['town_median_price'] = X_tr['town'].map(fold_town_map).fillna(global_med_tr)
    X_va['town_median_price'] = X_va['town'].map(fold_town_map).fillna(global_med_tr)

    # XGBoost
    xgb_pipe = Pipeline([('pre', preprocessor), ('model', XGBRegressor(**XGB_PARAMS))])
    xgb_pipe.fit(X_tr, y_tr_log)
    xgb_oof[va_idx]       = xgb_pipe.predict(X_va)
    xgb_test_folds[fold]  = xgb_pipe.predict(X_test)
    xgb_fold_rmse         = rmse(y_va, np.expm1(xgb_oof[va_idx]))
    fold_rmses_xgb.append(xgb_fold_rmse)

    # LightGBM
    lgbm_pipe = Pipeline([('pre', preprocessor), ('model', LGBMRegressor(**LGBM_PARAMS))])
    lgbm_pipe.fit(X_tr, y_tr_log)
    lgbm_oof[va_idx]      = lgbm_pipe.predict(X_va)
    lgbm_test_folds[fold] = lgbm_pipe.predict(X_test)
    lgbm_fold_rmse        = rmse(y_va, np.expm1(lgbm_oof[va_idx]))
    fold_rmses_lgbm.append(lgbm_fold_rmse)

    # Extra Trees
    et_pipe = Pipeline([('pre', preprocessor), ('model', ExtraTreesRegressor(**ET_PARAMS))])
    et_pipe.fit(X_tr, y_tr_log)
    et_oof[va_idx]       = et_pipe.predict(X_va)
    et_test_folds[fold]  = et_pipe.predict(X_test)
    et_fold_rmse         = rmse(y_va, np.expm1(et_oof[va_idx]))
    fold_rmses_et.append(et_fold_rmse)

    print(f'{fold+1:>5}  ${xgb_fold_rmse:>10,.0f}  ${lgbm_fold_rmse:>10,.0f}  ${et_fold_rmse:>10,.0f}')

print('-' * 50)
print(f'{"Mean":>5}  ${np.mean(fold_rmses_xgb):>10,.0f}  ${np.mean(fold_rmses_lgbm):>10,.0f}  ${np.mean(fold_rmses_et):>10,.0f}')

xgb_test_avg  = xgb_test_folds.mean(axis=0)
lgbm_test_avg = lgbm_test_folds.mean(axis=0)
et_test_avg   = et_test_folds.mean(axis=0)

print(f'\nOOF generation complete.')

---
## 8. Sanity Check — Compare Individual Models & Blends

In [ ]:
# Equal-weight 3-model blend
blend_3 = np.expm1((xgb_oof + lgbm_oof + et_oof) / 3)
print(f'OOF 3-model equal blend RMSE: ${rmse(y, blend_3):>8,.0f}')

# 2-model blend (same as v7 baseline)
blend_2_50 = np.expm1(0.5 * xgb_oof + 0.5 * lgbm_oof)
print(f'OOF 2-model 50/50 blend RMSE: ${rmse(y, blend_2_50):>8,.0f}')

# Individual
print(f'\nIndividual OOF RMSE:')
print(f'  XGB:  ${rmse(y, np.expm1(xgb_oof)):>8,.0f}')
print(f'  LGBM: ${rmse(y, np.expm1(lgbm_oof)):>8,.0f}')
print(f'  ET:   ${rmse(y, np.expm1(et_oof)):>8,.0f}')

print(f'\nv7 OOF RMSE (for comparison): $21,841')

---
## 9. Ridge Meta-Model on 3 OOF Features

Ridge now learns optimal weights for **3 base model predictions**.
With 3 meta-features, Ridge can:
- Assign higher weight to whichever model is most accurate overall
- Reduce influence of ET if it's weaker than XGB/LGBM
- Find any linear combination that improves on 2-model blending

In [ ]:
meta_X_train = np.column_stack([xgb_oof, lgbm_oof, et_oof])
meta_X_test  = np.column_stack([xgb_test_avg, lgbm_test_avg, et_test_avg])

print(f'Ridge alpha sweep (3 meta-features):')
print(f'{"alpha":>10}  {"RMSE":>12}  {"XGB coef":>10}  {"LGBM coef":>10}  {"ET coef":>10}')
print('-' * 62)

best_meta_rmse   = float('inf')
best_alpha_ridge = 1.0

for alpha in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
    ridge = Ridge(alpha=alpha)
    ridge.fit(meta_X_train, y_log)
    meta_pred_log = ridge.predict(meta_X_train)
    meta_rmse_val = rmse(y, np.expm1(meta_pred_log))
    print(f'{alpha:>10.3f}  ${meta_rmse_val:>10,.0f}  {ridge.coef_[0]:>10.4f}  {ridge.coef_[1]:>10.4f}  {ridge.coef_[2]:>10.4f}')
    if meta_rmse_val < best_meta_rmse:
        best_meta_rmse   = meta_rmse_val
        best_alpha_ridge = alpha

print(f'\nBest Ridge alpha: {best_alpha_ridge}')

meta_model = Ridge(alpha=best_alpha_ridge)
meta_model.fit(meta_X_train, y_log)
print(f'Learned coefficients:')
print(f'  XGB:  {meta_model.coef_[0]:.4f}')
print(f'  LGBM: {meta_model.coef_[1]:.4f}')
print(f'  ET:   {meta_model.coef_[2]:.4f}')
print(f'  Intercept: {meta_model.intercept_:.4f}')
print(f'\nOOF RMSE (meta-train, optimistic): ${best_meta_rmse:,.0f}')
print(f'v7 OOF RMSE:                         $21,841')
print(f'Change vs v7:                        ${best_meta_rmse - 21841:+,.0f}')

---
## 10. Generate Submission v8

In [ ]:
final_log  = meta_model.predict(meta_X_test)
final_pred = np.expm1(final_log)

sample_sub = pd.read_csv('../../outputs/submissions/sample_sub_reg.csv')
sub_v8 = pd.DataFrame({'Id': test['id'], 'Predicted': final_pred})
sub_v8 = sub_v8.set_index('Id').reindex(sample_sub['Id']).reset_index()

out = '../../outputs/submissions/sub_v8_et_stack.csv'
sub_v8.to_csv(out, index=False)
print(f'Saved: {out}')
print(f'Shape: {sub_v8.shape}')
print(f'Price range: ${sub_v8.Predicted.min():,.0f} – ${sub_v8.Predicted.max():,.0f}')
print(f'Price mean:  ${sub_v8.Predicted.mean():,.0f}')
print(f'\nv7 mean for comparison: ~$449,955')

---
## 11. Summary

### What changed vs v7
| Change | Detail |
|---|---|
| **3rd base model** | `ExtraTreesRegressor` added to OOF loop |
| **Meta-features** | Ridge trains on `[xgb_oof, lgbm_oof, et_oof]` instead of 2 columns |
| **Diversity** | ET (bagging, random splits) makes different errors than XGB/LGBM (boosting) |

### Score tracker
| Version | Model | OOF RMSE | Kaggle |
|---|---|---|---|
| v5 | Blend 45/55 | $21,570 | 22,428 |
| v6 | Log blend + OOF encoding | $21,818 | 22,124 |
| v7 | Stack XGB+LGBM | $21,841 | 21,906 |
| **v8** | **Stack XGB+LGBM+ET** | **_(run above)_** | **_(submit)_** |

### Key question: did ET help?
Check the Ridge coefficient for ET (`coef_[2]`) above:
- **Positive and sizeable (> 0.05)** → ET adds genuine signal; v8 likely improves on v7
- **Near zero (< 0.02)** → ET and XGB/LGBM make similar errors; Ridge ignores it
- **Negative** → Ridge corrects by going against ET; unusual but sometimes valid

### Next steps if v8 improves
- [ ] Try **Optuna** for hyperparameter tuning — Bayesian optimisation, smarter than RandomizedSearchCV
- [ ] Add `block_num` encoding: `pd.to_numeric(df['block'].str.extract(r'(\d+)')[0])`
- [ ] Feature selection — remove noisy features that hurt ET (tree-based models are sensitive to high-cardinality noise)